# Session 16: Fine-Tune a Support Ticket LLM

Fine-tune [Qwen2.5-0.5B-Instruct](https://huggingface.co/Qwen/Qwen2.5-0.5B-Instruct) on fictional support tickets using Hugging Face Transformers and PEFT. Compare the same model before and after training on held-out tickets.

## 1. Select the runtime and method

In Colab, select **Runtime → Change runtime type → T4 GPU**, then set `METHOD` to `"lora"` or `"qlora"`.

The setup cell installs the Hugging Face packages. Restart the kernel before changing methods. For a short CPU run, set `METHOD = "lora"` and `QUICK_RUN = True`.

In [ ]:
import subprocess
import sys

import torch

METHOD = "lora"  # Set to "qlora" on a T4 to load 4-bit weights.
QUICK_RUN = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if METHOD not in {"lora", "qlora"}:
    raise ValueError('Set METHOD to "lora" or "qlora".')
if device.type == "cpu" and (METHOD == "qlora" or not QUICK_RUN):
    raise RuntimeError('Select a Colab T4, or set METHOD = "lora" and QUICK_RUN = True.')

packages = ["transformers==4.57.1", "accelerate==1.10.1", "peft==0.17.1", "pandas"]
if METHOD == "qlora":
    packages.append("bitsandbytes==0.48.1")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *packages], check=True)

print(f"Device: {device} | Method: {METHOD}")

## 2. Load the data

The CSV contains 320 fictional tickets. Use the original training split to fit the adapter and the test split to compare results. The validation split is reserved for future tuning. The cell reads the published CSV. To read your local copy, set `TICKET_CSV` to `../sample_data/support_tickets.csv` when running from the practice folder.

Quick mode uses two tickets per category in each split. Use the full split for the class comparison.

In [ ]:
import pandas as pd

TICKET_CSV = "https://raw.githubusercontent.com/wlos/AI_SS/main/16-LLM-Fine-Tuning-and-Agents/sample_data/support_tickets.csv"
tickets = pd.read_csv(TICKET_CSV)

CATEGORIES = ("Account access", "Hardware", "Network", "Software")
# Keep the original test tickets separate from training.
train_rows = tickets.loc[tickets["split"] == "train"]
test_rows = tickets.loc[tickets["split"] == "test"]

if QUICK_RUN:
    # Keep two examples of each category in each split.
    train_rows = train_rows.groupby("category").head(2)
    test_rows = test_rows.groupby("category").head(2)

train_tickets = train_rows.to_dict("records")
test_tickets = test_rows.to_dict("records")
print(f"{len(train_tickets)} training tickets and {len(test_tickets)} test tickets.")

## 3. Load the model and tokenizer

Transformers loads the same instruct checkpoint for both methods. LoRA uses floating-point weights. QLoRA uses bitsandbytes to load the frozen linear-layer weights in 4-bit NF4.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, set_seed

CHECKPOINT = "Qwen/Qwen2.5-0.5B-Instruct"
SEED = 16
set_seed(SEED)

tokenizer = AutoTokenizer.from_pretrained(CHECKPOINT, padding_side="left")
weight_dtype = torch.float16 if device.type == "cuda" else torch.float32
model_options = {"dtype": weight_dtype}

# Quantise the frozen weights only for the QLoRA run.
if METHOD == "qlora":
    model_options["quantization_config"] = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=weight_dtype,
    )
    model_options["device_map"] = {"": torch.cuda.current_device()}

model = AutoModelForCausalLM.from_pretrained(CHECKPOINT, **model_options)
if METHOD == "qlora":
    from peft import prepare_model_for_kbit_training

    # Use the same frozen-weight preparation for the baseline and training.
    model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=False)
else:
    model.to(device)

## 4. Prepare the training examples

Use the model’s chat template for each ticket prompt. During supervised fine-tuning, the expected JSON answer and the end-of-turn token contribute to the loss. Use `-100` to mask the prompt. The Hugging Face data collator uses the same value for padded labels.

In [ ]:
import json

from transformers import DataCollatorForTokenClassification


def make_prompt(ticket):
    instruction = (
        f'Classify this support ticket as one of {", ".join(CATEGORIES)}. '
        'Return only a JSON object with the key "category".\n'
        f'Ticket: {ticket["text"]}'
    )
    return tokenizer.apply_chat_template(
        [{"role": "user", "content": instruction}],
        tokenize=False,
        add_generation_prompt=True,
    )


def make_training_example(ticket):
    prompt_ids = tokenizer.encode(make_prompt(ticket), add_special_tokens=False)
    answer = json.dumps({"category": ticket["category"]})
    answer_ids = tokenizer.encode(answer, add_special_tokens=False) + [tokenizer.eos_token_id]
    # Mask the prompt so only the expected answer contributes to the loss.
    return {
        "input_ids": prompt_ids + answer_ids,
        "labels": [-100] * len(prompt_ids) + answer_ids,
    }


training_examples = [make_training_example(ticket) for ticket in train_tickets]
collate_training_batch = DataCollatorForTokenClassification(tokenizer, label_pad_token_id=-100)

## 5. Test the model before fine-tuning

Generate a reply for every test ticket. Greedy decoding uses the highest-scoring token at each step. Keep these prompts and generation settings for the final comparison.

In [ ]:
from transformers import GenerationConfig

BATCH_SIZE = 4 if device.type == "cuda" else 2
generation_settings = GenerationConfig(
    do_sample=False,
    max_new_tokens=32,
    eos_token_id=model.generation_config.eos_token_id,
    pad_token_id=tokenizer.pad_token_id,
)


@torch.inference_mode()
def generate_replies(language_model, prompts):
    language_model.eval()
    replies = []
    for start in range(0, len(prompts), BATCH_SIZE):
        batch = tokenizer(
            prompts[start:start + BATCH_SIZE],
            padding=True,
            add_special_tokens=False,
            return_tensors="pt",
        ).to(language_model.device)
        generated = language_model.generate(**batch, generation_config=generation_settings)
        # The returned sequence includes the prompt. Decode only new tokens.
        reply_tokens = generated[:, batch["input_ids"].shape[1]:]
        replies.extend(tokenizer.batch_decode(reply_tokens, skip_special_tokens=True))
    return replies


test_prompts = [make_prompt(ticket) for ticket in test_tickets]
before_replies = generate_replies(model, test_prompts)

## 6. Fine-tune with LoRA or QLoRA

PEFT freezes the model weights and attaches a small adapter. QLoRA prepares the quantised model before adding the same type of adapter. Hugging Face Trainer runs one pass over the selected training tickets. Inspect the trainable parameter count before training starts.

In [ ]:
from peft import LoraConfig, get_peft_model
from transformers import Trainer, TrainingArguments

# Attach a trainable adapter to the frozen checkpoint.
adapter_model = get_peft_model(
    model,
    LoraConfig(
        task_type="CAUSAL_LM",
        r=8,
        lora_alpha=16,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    ),
)
adapter_model.print_trainable_parameters()

training_args = TrainingArguments(
    output_dir="session16-training",
    num_train_epochs=1,
    per_device_train_batch_size=BATCH_SIZE,
    learning_rate=2e-4,
    optim="adamw_torch",
    fp16=device.type == "cuda",
    use_cpu=device.type == "cpu",
    dataloader_pin_memory=device.type == "cuda",
    remove_unused_columns=False,
    logging_steps=10,
    save_strategy="no",
    report_to="none",
    seed=SEED,
)
# Training uses the full sequence at once, so it does not need the generation cache.
adapter_model.config.use_cache = False
trainer = Trainer(
    model=adapter_model,
    args=training_args,
    train_dataset=training_examples,
    data_collator=collate_training_batch,
    processing_class=tokenizer,
)
trainer.train()
adapter_model.config.use_cache = True

## 7. Test and compare

Evaluate the adapted model on the same tickets. **JSON format pass rate** measures replies with exactly the expected key and an allowed category. **Category accuracy** measures correct labels across all test tickets, counting invalid replies as incorrect.

The examples show tickets where either reply has an issue. JSON wrapped in a Markdown code fence counts as a format issue even when it contains the expected category.

In [ ]:
def read_category(reply):
    try:
        answer = json.loads(reply)
    except json.JSONDecodeError:
        return None
    if isinstance(answer, dict) and set(answer) == {"category"}:
        category = answer["category"]
        if category in CATEGORIES:
            return category
    return None


def reply_status(reply, expected):
    category = read_category(reply)
    if category is None:
        return "Invalid format"
    return "Correct" if category == expected else "Wrong category"


def score_replies(replies):
    parsed = [read_category(reply) for reply in replies]
    matches = sum(
        category == ticket["category"]
        for category, ticket in zip(parsed, test_tickets, strict=True)
    )
    return {
        "JSON format pass rate": sum(category is not None for category in parsed) / len(test_tickets),
        "Category accuracy": matches / len(test_tickets),
    }


# Reuse the baseline prompts and decoding settings for a paired comparison.
after_replies = generate_replies(adapter_model, test_prompts)
before_scores = score_replies(before_replies)
after_scores = score_replies(after_replies)

print(f'{"Metric":<24} {"Before":>8} {"After":>8}')
for metric, before in before_scores.items():
    print(f"{metric:<24} {before:>8.1%} {after_scores[metric]:>8.1%}")

examples_with_issues = [
    (ticket, before, after)
    for ticket, before, after in zip(test_tickets, before_replies, after_replies, strict=True)
    if read_category(before) != ticket["category"] or read_category(after) != ticket["category"]
]
for ticket, before, after in examples_with_issues[:2]:
    expected = ticket["category"]
    print(f'\nTicket: {ticket["text"]}\nExpected: {expected}')
    print(f"Before ({reply_status(before, expected)}): {before!r}")
    print(f"After ({reply_status(after, expected)}): {after!r}")

### Export the adapter

Run this cell after fine-tuning. In Colab, download `session16-ticket-adapter.zip` from the **Files** pane. The ZIP contains the adapter weights and PEFT configuration, plus the training method for loading the original checkpoint with matching quantisation.

In [ ]:
from tempfile import TemporaryDirectory
from zipfile import ZIP_DEFLATED, ZipFile

ADAPTER_ZIP = "session16-ticket-adapter.zip"
# Package the adapter weights and settings for the agent notebook.
with TemporaryDirectory() as adapter_dir:
    adapter_model.save_pretrained(adapter_dir, safe_serialization=True)
    with ZipFile(ADAPTER_ZIP, "w", compression=ZIP_DEFLATED) as archive:
        for filename in ("adapter_config.json", "adapter_model.safetensors"):
            archive.write(f"{adapter_dir}/{filename}", filename)
        archive.writestr("training_method.txt", METHOD)

print("Download:", ADAPTER_ZIP)